# 旧六线方向分类版：直接移植11项既有优化

独立MA_DIRECTION_REFERENCE、原完整入场与uncapped出场、无V2。不是ER＋V2，也不是FIXED5/OHLC。复用MA自己的12份基准，只新算缺失账户。2026连续覆盖至北京时间11月15日全天。零费、零资金费、退出滑点0.02%。不重做ER/MA比较。

请在独立运行时顺序运行1→6；不要替换正在运行的ER Notebook。市场Drive缓存可共享，结果与断点完全独立。截图2020的112.88%旧窗口尚未找回，本任务采用登记中统一2月16日起的真实MA实现与账本，不保证复现截图数字。

## 1. CPU环境与必要依赖（GPU不用于金融逻辑）

In [ ]:
import sys,os,subprocess,shutil,platform
assert sys.version_info >= (3,12), '请使用Python3.12或更高的Colab运行时'
subprocess.run([sys.executable,'-m','pip','-q','install','numpy==2.3.5','pandas==3.0.1','psutil==7.0.0'],check=True)
import psutil
print('Python:',sys.version.split()[0], 'CPU:',os.cpu_count(),platform.machine())
print('RAM available/total GB:',round(psutil.virtual_memory().available/2**30,1),round(psutil.virtual_memory().total/2**30,1))
print('Local disk free GB:',round(shutil.disk_usage('/content').free/2**30,1))
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout)
else: print('GPU: none. This financial replay uses CPU.')

## 2. 固定提交与私有仓库认证
在Colab Secrets保存GITHUB_TOKEN，或使用隐藏输入。凭据不保存在Notebook、输出或remote URL。私有Notebook若打不开，先授权Colab访问GitHub，或下载.ipynb上传Colab。

In [ ]:
from pathlib import Path
import tempfile,getpass,json
CODE_COMMIT = 'ec73e9c2a791d2106f06453e60ef8941fab432c5'
REPO = Path('/content/ma-direction-code-'+CODE_COMMIT[:7])
if (REPO/'.git').exists():
    assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==CODE_COMMIT, '目录提交不一致；请使用新目录，不覆盖旧任务'
else:
    if REPO.exists() and any(REPO.iterdir()): raise RuntimeError('目标目录非空；换一个新目录')
    try:
        from google.colab import userdata
        token=userdata.get('GITHUB_TOKEN')
    except Exception:
        token=getpass.getpass('GitHub private repository token (hidden): ')
    if not token: raise RuntimeError('缺少私有仓库读取凭据')
    env=os.environ.copy();env['COLAB_GH_TOKEN']=token;env['GIT_TERMINAL_PROMPT']='0'
    with tempfile.TemporaryDirectory() as tmp:
        ask=Path(tmp)/'askpass.py'
        ask.write_text('#!/usr/bin/env python3\nimport os,sys\nprint("x-access-token" if "username" in sys.argv[1].lower() else os.environ["COLAB_GH_TOKEN"])\n')
        ask.chmod(0o700);env['GIT_ASKPASS']=str(ask)
        try:
            REPO.mkdir(parents=True,exist_ok=True)
            for cmd in [['git','init','-q'],['git','remote','add','origin','https://github.com/HITGoose/btc-eth-backtest.git'],['git','-c','credential.helper=','fetch','-q','--depth=1','origin',CODE_COMMIT],['git','checkout','-q','--detach','FETCH_HEAD']]:
                subprocess.run(cmd,cwd=REPO,env=env,check=True,capture_output=True)
        finally:
            env.pop('COLAB_GH_TOKEN',None);del token
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==CODE_COMMIT
os.environ['BTC_REPO_ROOT']=str(REPO)
ENTRY=REPO/'outputs/ma_direction_retests'
print('Locked implementation:',CODE_COMMIT)
print('Repository:',REPO)

## 3. Drive与行情缓存

先完成Drive授权；失败时重试本格，不跳过备份启动长任务。原始逐笔不全在Git中，缺失从币安官方获取并校验。复制到本地计算。只恢复MA同提交结果，不导入ER账本或断点。

In [ ]:
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    try: drive.mount('/content/drive')
    except ValueError: raise RuntimeError('Drive挂载失败：完成授权并重试本格；不要跳过备份。') from None
DRIVE=Path('/content/drive/MyDrive')
DRIVE_DATA=DRIVE/'btc_eth_market_cache'
DRIVE_RESULTS=DRIVE/'btc_eth_ma_direction_optimizations'/CODE_COMMIT[:7]
DATA=Path('/content/btc_eth_market_cache')
RESULTS=Path('/content/ma_direction_results_'+CODE_COMMIT[:7])
def restore(source,target,prior_only=False):
    if not source.exists(): return
    for p in source.rglob('*'):
        if not p.is_file() or p.name.endswith(('.copying','.tmp','.part','.restoring')): continue
        if prior_only and (p.suffix not in ['.json','.npz'] or 'smoke' in p.parts): continue
        dest=target/p.relative_to(source)
        if dest.exists() and dest.stat().st_mtime>=p.stat().st_mtime and dest.stat().st_size==p.stat().st_size: continue
        dest.parent.mkdir(parents=True,exist_ok=True)
        temp=dest.with_suffix(dest.suffix+'.restoring');shutil.copy2(p,temp);temp.replace(dest)
DATA.mkdir(parents=True,exist_ok=True);RESULTS.mkdir(parents=True,exist_ok=True)
restore(DRIVE_DATA,DATA);restore(DRIVE_RESULTS,RESULTS)
os.environ.update(BTC_DATA_ROOT=str(DATA),BTC_OUTPUT_ROOT=str(RESULTS),BTC_DRIVE_DATA=str(DRIVE_DATA),BTC_DRIVE_RESULTS=str(DRIVE_RESULTS),PYTHONUNBUFFERED='1')
os.environ.pop('BTC_IMPORT_ROOT',None);os.environ.pop('BTC_IMPORT_ZIP',None)
print('Local:',DATA,RESULTS)
print('Drive checkpoints:',DRIVE_RESULTS)
print('MA自己的12份基准随代码提供；没有年度计算启动。')


## 4. 8小时最小检查

核对真实非V2调用链、REF分钟曲线与MA旧账本完全一致，以及ATR、撤单、阶梯、轻仓机制。不是年度结果；成功后运行5。

In [ ]:
env=os.environ.copy()
env['BTC_OUTPUT_ROOT']=str(RESULTS/'short_validation')
env['BTC_DRIVE_RESULTS']=str(DRIVE_RESULTS/'short_validation')
subprocess.run([sys.executable,'-u',str(ENTRY/'smoke.py')],cwd=REPO,env=env,check=True)
print('Only short checks complete. Full pipeline has NOT started.')

## 5. 11项顺序执行

先补2026 MA基准。各项完成即保存年度表。训练网格只用2020—2022，冻结后评估；不采用ER优化的赢家。每30秒进度，失败项保留缺口，其他独立项继续。停止后按1→3再5恢复：相同指纹已完成任务跳过，未完成从兼容日断点恢复。不改代码、不删除指纹检查。

In [ ]:
import signal
# 可选仅降低并行数：os.environ['BTC_WORKERS']='1'
process=subprocess.Popen([sys.executable,'-u',str(ENTRY/'pipeline.py')],cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
try:
    for line in process.stdout: print(line,end='',flush=True)
    if process.wait()!=0: raise RuntimeError('Pipeline interrupted/blocked; see logs. Saved checkpoints retained.')
except KeyboardInterrupt:
    try: os.killpg(process.pid,signal.SIGTERM)
    except ProcessLookupError: pass
    process.wait()
    print('Stopped this process group. Resume same code and output root.',flush=True)
    raise
print('See incomplete.csv / stage_logs for any unresolved item.')

Frozen MA_DIRECTION_REFERENCE, NO V2, ZERO; through BJT Nov15 inclusive.
Stage BASELINE_2026 new tasks 2 CPU workers 4
RUNNING BASELINE_2026 2026 BTC 2026-01-05T08:00:00.000+08:00; 2026 ETH 2026-01-04T08:00:00.000+08:00
RUNNING BASELINE_2026 2026 BTC 2026-01-14T08:00:00.000+08:00; 2026 ETH 2026-01-12T08:00:00.000+08:00
RUNNING BASELINE_2026 2026 BTC 2026-01-23T08:00:00.000+08:00; 2026 ETH 2026-01-20T08:00:00.000+08:00
RUNNING BASELINE_2026 2026 BTC 2026-02-08T08:00:00.000+08:00; 2026 ETH 2026-02-02T08:00:00.000+08:00
RUNNING BASELINE_2026 2026 BTC 2026-02-11T08:00:00.000+08:00; 2026 ETH 2026-02-28T08:00:00.000+08:00
RUNNING BASELINE_2026 2026 BTC 2026-02-20T08:00:00.000+08:00; 2026 ETH 2026-03-21T08:00:00.000+08:00
RUNNING BASELINE_2026 2026 BTC 2026-03-02T08:00:00.000+08:00; 2026 ETH 2026-03-31T08:00:00.000+08:00
RUNNING BASELINE_2026 2026 BTC 2026-03-13T08:00:00.000+08:00; 2026 ETH 2026-04-10T08:00:00.000+08:00
RUNNING BASELINE_2026 2026 BTC 2026-03-18T08:00:00.000+08:00; 2026 ETH 20

## 6. 汇总及下载

只读取已完成账本，不启动账户回放。ZIP完成CRC核验并生成SHA收据，复制到Drive。如果聊天上传ZIP连接失效，运行7单独下载报告与CSV；这不等于ZIP损坏。

In [ ]:
# Only read saved results; this command does NOT run pipeline.main or accounts.
script="import sys;sys.path.insert(0,"+repr(str(ENTRY))+");import common as c;r=c.module('export_retest_report',c.HERE/'report.py');r.refresh();p=c.module('export_retest_pipeline',c.HERE/'pipeline.py');p.export()"
subprocess.run([sys.executable,'-u','-c',script],cwd=REPO,check=True)
print((RESULTS/'REPORT.md').read_text())
from google.colab import files
files.download(str(RESULTS/'ma_retests_results_for_review.zip'))

## 7. 单文件下载（ZIP传输失败时）
修改FILE后重复运行，优先REPORT.md、annual.csv、period_summary.csv、incomplete.csv；需要归因再传trade_matching.csv。

In [ ]:
FILE='annual.csv'
from google.colab import files
p=RESULTS/FILE
assert p.is_file(), '此文件尚未生成；先运行6'
files.download(str(p))
